# A4：交叉口让行 —— LTAP-OD 左转让行闭环测试
# A4: Intersection Yield — LTAP-OD Closed-Loop Test

## 一、实验原理 | Principle

### 1.1 LTAP-OD 是什么 | What is LTAP-OD
**LTAP-OD = Left Turn Across Path – Opposite Direction（左转穿越对向车流）**，
是城市交叉口最危险的场景之一（占交叉口事故约 30%）：
- Ego 自车北向行驶，准备**左转**（向西）
- Target 对向车辆南向**直行**，拥有路权
- Ego 的左转轨迹与 Target 的直行车道存在一个**冲突点（Conflict Point）**

```
        Target (南向直行, 16 m/s)
            |
            v
  ----------+-----------   <- 东西向道路
            |
      C(98.5, 0)  <- 冲突点
            |
  Ego ------+           Ego 北向行驶, 在此左转
   (北向 9 m/s)
```

### 1.2 冲突点 TTC | Conflict-Point TTC
与跟驰场景的 TTC（相对距离/相对速度）不同，交叉口场景使用**冲突点 TTC**：

- $t_{ego} = d_{ego} / v_{ego}$：自车到达冲突点的预计时间
- $t_{tgt} = d_{tgt} / v_{tgt}$：目标车到达冲突点的预计时间
- **让行判据**：若 $t_{tgt} < t_{ego} + t_{margin}$（本实验 $t_{margin}=1.5s$），
  说明目标车先到 → 自车必须**停车让行**，否则存在碰撞风险

### 1.3 PET（后侵入时间）| Post-Encroachment Time
**PET** 是交叉口安全评价的金标准（SSAM、FHWA 推荐）：

$$PET = |t_{ego}^{pass} - t_{tgt}^{pass}|$$

其中 $t^{pass}$ 是车辆中心通过冲突点的时刻。
- PET 越大越安全；**PET > 1.0s** 为本实验的通过线
- PET 与"是否碰撞"是互补指标：**没撞 ≠ 安全**（见对比实验）

### 1.4 让行策略 | Yield Strategy
本实验的参考控制器是四状态机：
| 状态 | 条件 | 动作 |
|------|------|------|
| assess（评估） | t < 0.5s | 直行，计算双方到达时间 |
| yield（让行） | $t_{tgt} < t_{ego} + 1.5$ | 按 $v^2=2ad$ 减速至停车线 |
| turn（转向） | 目标车已通过冲突点 | pure-pursuit 跟踪左转航点 |
| done（完成） | 到达终点 | 直行巡航 |

## 二、实验目的 | Objectives
1. 理解 LTAP-OD 场景的冲突点建模方法
2. 掌握冲突点 TTC 与 PET 的计算及其物理意义
3. 实现"评估-让行-转向"状态机控制器
4. 通过对比实验理解"无碰撞 ≠ 安全"（PET 的价值）

## 三、学生任务 | Your Tasks
1. **阅读** `YieldController`，理解四状态机的切换条件
2. **运行**让行测试，确认 `collision=False` 且 `PET > 1.0s`
3. **运行**对比实验（`NaiveTurnController`），观察 PET 如何恶化
4. **调参**：把 `margin` 从 1.5 改为 0.5 —— 还安全吗？PET 变为多少？
5. **思考题**：如果 Target 车速提高到 20 m/s，让行逻辑还成立吗？需要调整哪个参数？
6. **挑战**：实现基于恒定减速度假设（而非匀速假设）的 $t_{tgt}$ 估计

## 四、验证内容 | Verification
- [ ] esmini 成功初始化交叉口场景（2 个交通参与者）
- [ ] 让行测试：`assert not collision` 通过
- [ ] 让行测试：`assert pet > 1.0` 通过（PET > 1.0s）
- [ ] 对比测试：naive 控制器 PET < 1.0s 或发生碰撞（证明让行的必要性）
- [ ] 轨迹图显示 Ego 在停车线前停稳、Target 通过后才起步左转

## 五、结果分析 | Result Analysis

### 5.1 关键指标解读 | Key Metrics

| 指标 | 通过线 | 物理意义 |
|------|--------|---------|
| `collision` | 必须为 False | 是否发生碰撞——底线 |
| `pet` | > 1.0s | **PET（Post-Encroachment Time，后侵入时间）** = 两车通过冲突点的时刻差。这是交叉口安全的金标准 |
| `naive_unsafe` | 必须为 True | 对比实验（不让行）必须不安全——证明让行的必要性 |

**PET 的物理意义**：假设 Target 在 t=10.0s 通过冲突点，Ego 在 t=12.5s 通过，则 PET = 2.5s。PET 越大，说明两车在冲突点的时间间隔越大，越安全。

**为什么"无碰撞 ≠ 安全"**：naive 控制器可能以 0.15s 的 PET"惊险"通过——没撞，但只要 Target 快 0.2s 或 Ego 快 0.2s 就撞了。PET < 1.0s 在 SSAM（Surrogate Safety Assessment Model）中被记为一次"严重冲突（serious conflict）"。**好的自动驾驶不仅要不撞，还要留足裕度**。

**PET vs TTC**：跟驰场景用 TTC（纵向相对运动），交叉口用 PET（时空分离）。两者都是"安全裕度"的量化，但适用场景不同——这是本实验要建立的核心概念区分。

### 5.2 让行判据的物理意义 | The Yield Criterion

**判据**：若 $t_{tgt} < t_{ego} + margin$（margin=1.5s），则让行。

- $t_{tgt} = d_{tgt}/v_{tgt}$：目标车到达冲突点的预计时间（匀速假设）
- $t_{ego} = d_{ego}/v_{ego}$：自车到达冲突点的预计时间
- $margin = 1.5s$：安全裕度。即使估计有误差，1.5s 的缓冲也能吸收

**margin 的整定**：margin 越大越安全，但等待时间越长（通行效率低）。1.5s 是经验值：Target 16m/s × 1.5s = 24m，相当于多让出 24 米的距离。城市交叉口人类驾驶员的让行裕度约 1.0~2.0s。

### 5.3 结果合理性判断 | Is the Result Reasonable?

- **PET ≈ 2.0~2.5s，无碰撞，naive 对比 PET < 0.5s**：理想。让行逻辑工作正常，对比效果显著。
- **PET 刚好 > 1.0s（如 1.1s）**：临界通过。margin=1.5s 可能偏小，或减速曲线不够激进。真实传感器噪声下可能跌破 1.0s。
- **PET 很大（> 4s）**：过于保守。Ego 等待太久，影响通行效率。可以尝试 margin=1.0s。
- **naive 对比居然安全（naive_unsafe=False）**：场景设置有问题——可能 Target 速度太慢或初始距离太远，导致即使不让行也不会冲突。检查场景参数。

### 5.4 四状态机的时序特征 | State Machine Timing

典型的成功时序：
1. **assess（0~0.5s）**：Ego 直行，计算 $t_{tgt}$ 和 $t_{ego}$，发现 $t_{tgt} < t_{ego} + 1.5$ → 切换到 yield
2. **yield（0.5s~Target通过）**：按 $v²=2ad$ 减速，在停车线前停稳。速度曲线降为 0 并保持
3. **turn（Target通过后）**：pure-pursuit 跟踪左转航点，横向位置从 y=-30 逐渐变为 y=5.5
4. **done**：到达终点，直行巡航

如果速度曲线在 yield 阶段没有降为 0，说明让行逻辑没触发——检查判据中的 margin 或 $t_{tgt}$ 计算。

### 5.5 误差来源分析 | Error Sources

1. **匀速假设**：$t_{tgt} = d_{tgt}/v_{tgt}$ 假设 Target 匀速。如果 Target 在接近冲突点时加速（抢行），实际到达时间更早，PET 会被高估。挑战任务 6 让学生实现恒定减速度假设的改进版。
2. **pure-pursuit 跟踪误差**：左转航点是离散的，pure-pursuit 在曲率大的地方有切角（corner-cutting），实际轨迹比航点更"直"，可能侵入对向车道。预瞄距离（lookahead）的整定很关键。
3. **目标车冻结**：同 A1-A3，Target 的直行运动在 ctypes 模式下由 Python 端驱动，esmini 只做场景加载。
4. **冲突点位置**：C(98.5, 0) 是几何计算值，实际车辆有宽度（~1.8m），"通过冲突点"的判定用的是车辆中心，有约 ±1m 的不确定性。

### 5.6 与理论预期的对比 | vs. Theory

| 理论/行业惯例 | 本实验 | 对比结论 |
|-------------|--------|---------|
| SSAM：PET < 1.5s 为冲突，< 1.0s 为严重冲突 | 通过线 1.0s | 与 SSAM 严重冲突阈值一致 |
| 人类驾驶员让行裕度 1.0~2.0s | margin=1.5s | 取中间值，符合人类行为 |
| 真实 LTAP 需 V2X 或视觉预测对向车意图 | 本实验假设 Target 匀速直行 | 简化假设；真实系统需要意图预测（挑战任务方向） |


> ⚠️ **架构诚实声明**：本实验采用混合架构——Target车辆由esmini引擎加载，但实测发现esmini在ctypes模式下目标车的脚本动作不生效（车辆冻结不动，等价于静态障碍物）；Ego车辆由Python运动学模型（Kinematic Bicycle Model）积分推进，通过`SE_ReportObjectPos`上报给esmini。这不是完整的esmini双车闭环，而是"esmini场景加载 + Python Ego闭环"的教学简化。如需全动态双车，请使用esmini二进制headless模式。


> ⚠️ **Architecture Disclosure**: This experiment uses a hybrid architecture — the Target vehicle is loaded by the esmini engine, but testing revealed that scripted target actions do not take effect in ctypes mode (vehicle frozen, equivalent to a static obstacle); the Ego vehicle is integrated by a Python kinematic bicycle model and reported to esmini via `SE_ReportObjectPos`. This is a teaching simplification ("esmini scene loading + Python Ego closed-loop"), not a full esmini two-vehicle closed loop. For fully dynamic two-vehicle simulation, use the esmini binary in headless mode.



In [ ]:
import sys
sys.dont_write_bytecode = True  # Disable .pyc cache
# Clean stale .pyc cache (esmini ctypes is sensitive to it)
import shutil, os
for _p in ['/home/jovyan/labs/common/__pycache__']:
    if os.path.exists(_p):
        shutil.rmtree(_p, ignore_errors=True)
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import EsminiEnv
# Force reload (bypass ctypes deadlock caused by stale .pyc)
import common.esmini_env
from common.esmini_env import EsminiEnv
print('OK: labs/common imported')

### 常见报错说明 | Troubleshooting

| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `Scenario file not found!` | 场景文件路径不对 | 确认 `labs/scenarios/A_esmini/A4_intersection/` 下有 `intersection.xosc`；新版目录与旧版 `ad2026/` 做了兼容查找 |
| `esmini SE_InitWithArgs failed` | esmini 初始化失败 | 检查 xosc 中的 `filepath="intersection.xosc"` 是否与 xodr 文件名一致；确认在场景目录下运行 |
| `TypeError: 'tuple' object has no attribute 'get'` | 控制器返回了元组而非字典 | `step()` 必须返回 `dict`（如 `{'target_a': a, 'steer_angle': d}`），不能直接返回 `pure_pursuit()` 的元组 |
| 仿真卡住无响应 | 可能是 stale `.pyc` 导致 ctypes 死锁 | 首个代码 cell 已自动清理 `__pycache__` 并 `importlib.reload`；若仍卡住，重启 kernel 后重新运行 |
| `PET is None` | 某一辆车未通过冲突点 | 检查控制器状态机是否卡在 `yield`（目标车是否真的通过了）；或仿真步数不足 |
| 图中中文显示为方框 | 容器无中文字体 | 本实验所有图表标签使用英文，这是已知限制 |


## 1. 实现让行控制器 | Implement the Yield Controller

`YieldController` 实现四状态机（assess → yield → turn → done），
`NaiveTurnController` 为对比基线（无让行逻辑，直接左转）。


In [ ]:
import math

# 场景几何（与 scenario.yaml 一致）
CX, CY = 98.5, 0.0   # 冲突点 Conflict point
STOP_Y = -10.0        # 停车线 Stop line

class YieldController(BaseController):
    """LTAP-OD 让行控制器：评估 -> 让行 -> 左转 -> 完成."""

    def __init__(self, config=None):
        super().__init__(config)
        self.cx, self.cy = CX, CY
        self.stop_y = STOP_Y
        self.margin = self.config.get('margin', 1.5)
        # 左转航点（世界坐标）
        self.wps = [(101.5, -30.0), (101.5, -12.0), (100.5, -4.0),
                    (98.0, 1.5), (93.0, 4.5), (86.0, 5.5), (78.0, 5.5)]
        self.reset()

    def reset(self):
        self.phase = 'assess'
        self.t_tgt_pass = None
        self.t_ego_pass = None
        self._prev_tgt_y = None
        self._prev_ego_x = None
        self.min_conflict_ttc = float('inf')
        self.conflict_ttc = None  # t_ego when target enters conflict zone
        self._wp_idx = 0

    def _pure_pursuit(self, x, y, yaw, v, v_des):
        # snap to nearest waypoint (handles post-yield resume correctly)
        dists = [math.hypot(wx - x, wy - y) for wx, wy in self.wps]
        self._wp_idx = min(range(len(self.wps)), key=lambda i: dists[i])
        L = 4.0  # lookahead distance
        tx, ty = self.wps[-1]
        for i in range(self._wp_idx, len(self.wps)):
            wx, wy = self.wps[i]
            if math.hypot(wx - x, wy - y) >= L:
                tx, ty = wx, wy
                self._wp_idx = i
                break
        alpha = math.atan2(ty - y, tx - x) - yaw
        alpha = (alpha + math.pi) % (2 * math.pi) - math.pi
        delta = math.atan2(2 * 2.8 * math.sin(alpha), L)
        delta = max(-0.5, min(0.5, delta))
        a = max(-4.0, min(2.0, 1.5 * (v_des - v)))
        return a, delta

    def step(self, obs):
        ego = obs['ego']
        t, dt = obs['sim_time'], obs['dt']
        x, y, yaw, v = ego['x'], ego['y'], ego['yaw'], ego['v']
        tgts = obs.get('targets', [])
        t_ego = math.hypot(x - self.cx, y - self.cy) / max(v, 0.5)

        if tgts:
            tgt = tgts[0]
            ty, tv = tgt['y'], max(tgt['v'], 0.1)
            t_tgt = (ty - self.cy) / tv if ty > self.cy else 0.0
            if (self._prev_tgt_y is not None and self._prev_tgt_y > self.cy
                    and ty <= self.cy and self.t_tgt_pass is None):
                self.t_tgt_pass = t
                self.conflict_ttc = t_ego  # snapshot at entry instant
            self._prev_tgt_y = ty
        else:
            t_tgt = float('inf')

        # ego crosses the target lane (x decreasing through CX)
        if (self._prev_ego_x is not None and self._prev_ego_x > self.cx
                and x <= self.cx and self.t_ego_pass is None):
            self.t_ego_pass = t
        self._prev_ego_x = x

        if self.t_tgt_pass is None:
            self.min_conflict_ttc = min(self.min_conflict_ttc,
                                        min(t_ego, t_tgt))

        if self.phase == 'assess':
            a = max(-4.0, min(2.0, 1.5 * (9.0 - v)))
            if t >= 0.5:
                self.phase = ('yield' if (self.t_tgt_pass is None
                                          and t_tgt < t_ego + self.margin)
                              else 'turn')
            return {'target_a': a, 'steer_angle': 0.0}

        if self.phase == 'yield':
            if self.t_tgt_pass is not None:
                self.phase = 'turn'
                a_pp, d_pp = self._pure_pursuit(x, y, yaw, v, 3.0)
                return {'target_a': a_pp, 'steer_angle': d_pp}
            d_stop = self.stop_y - y
            if d_stop <= 0.3 and v < 0.3:
                return {'target_a': 0.0, 'steer_angle': 0.0}  # hold
            # v^2 = 2*a*d braking profile
            v_allow = math.sqrt(max(0.0, 2 * 3.5 * max(d_stop, 0.0)))
            a = max(-6.0, min(2.0, 2.0 * (min(v_allow, 4.0) - v)))
            return {'target_a': a, 'steer_angle': 0.0}

        if self.phase == 'turn':
            if math.hypot(x - self.wps[-1][0],
                           y - self.wps[-1][1]) < 3.0:
                self.phase = 'done'
            a_pp, d_pp = self._pure_pursuit(x, y, yaw, v, 6.0)
            return {'target_a': a_pp, 'steer_angle': d_pp}

        return {'target_a': 1.0 * (6.0 - v), 'steer_angle': 0.0}


class NaiveTurnController(BaseController):
    """对比基线：无让行逻辑，直接左转（预期：不安全）."""

    def __init__(self, config=None):
        super().__init__(config)
        self.cx, self.cy = CX, CY
        self.wps = [(101.5, -30.0), (101.5, -12.0), (100.5, -4.0),
                    (98.0, 1.5), (93.0, 4.5), (86.0, 5.5), (78.0, 5.5)]
        self.reset()

    def reset(self):
        self.t_tgt_pass = None
        self.t_ego_pass = None
        self._prev_tgt_y = None
        self._prev_ego_x = None
        self.min_conflict_ttc = float('inf')
        self.conflict_ttc = None
        self._wp_idx = 0

    def step(self, obs):
        ego = obs['ego']
        t, dt = obs['sim_time'], obs['dt']
        x, y, yaw, v = ego['x'], ego['y'], ego['yaw'], ego['v']
        tgts = obs.get('targets', [])
        t_ego = math.hypot(x - self.cx, y - self.cy) / max(v, 0.5)
        if tgts:
            tgt = tgts[0]
            ty, tv = tgt['y'], max(tgt['v'], 0.1)
            t_tgt = (ty - self.cy) / tv if ty > self.cy else 0.0
            if (self._prev_tgt_y is not None and self._prev_tgt_y > self.cy
                    and ty <= self.cy and self.t_tgt_pass is None):
                self.t_tgt_pass = t
                self.conflict_ttc = t_ego
            self._prev_tgt_y = ty
        else:
            t_tgt = float('inf')
        if (self._prev_ego_x is not None and self._prev_ego_x > self.cx
                and x <= self.cx and self.t_ego_pass is None):
            self.t_ego_pass = t
        self._prev_ego_x = x
        if self.t_tgt_pass is None:
            self.min_conflict_ttc = min(self.min_conflict_ttc,
                                        min(t_ego, t_tgt))
        # pure pursuit at constant speed, NO yield logic
        dists = [math.hypot(wx - x, wy - y) for wx, wy in self.wps]
        self._wp_idx = min(range(len(self.wps)), key=lambda i: dists[i])
        L = 4.0
        tx, ty = self.wps[-1]
        for i in range(self._wp_idx, len(self.wps)):
            wx, wy = self.wps[i]
            if math.hypot(wx - x, wy - y) >= L:
                tx, ty = wx, wy
                self._wp_idx = i
                break
        alpha = math.atan2(ty - y, tx - x) - yaw
        alpha = (alpha + math.pi) % (2 * math.pi) - math.pi
        delta = max(-0.5, min(0.5, math.atan2(2 * 2.8 * math.sin(alpha), L)))
        a = max(-4.0, min(2.0, 1.5 * (8.0 - v)))
        return {'target_a': a, 'steer_angle': delta}

print('OK: controllers defined')

## 2. 运行让行测试 | Yield Test (real esmini closed-loop)

使用 **真实 esmini 引擎**（ctypes 调用 `libesminiLib.so`）：
Target 由 esmini 脚本驾驶（南向直行 16 m/s），Ego 由上面的 Python 控制器驾驶。


In [ ]:
# Locate the A4 scenario files (new layout with fallback)
import os
candidates = [
    '/home/jovyan/labs/scenarios/A_esmini/A4_intersection/intersection.xosc',
    '/home/jovyan/labs/ad2026/intersection.xosc',
]
scenario_path = next((p for p in candidates if os.path.exists(p)), None)
assert scenario_path is not None, 'Scenario file not found! Check labs/scenarios/A_esmini/A4_intersection/'
print(f'OK: scenario = {scenario_path}')

In [ ]:
# Run yield test with real esmini closed-loop
import sys, math
sys.path.insert(0, '/home/jovyan/labs')
from common.esmini_env import EsminiEnv

controller = YieldController()
env = EsminiEnv(scenario=scenario_path, dt=0.05)
obs = env.reset(ego_x=101.5, ego_y=-30.0, ego_yaw=math.pi / 2, ego_v=9.0)
controller.reset()

traj, collision = [], False
min_gap = float('inf')

for step in range(400):
    action = controller.step(obs)
    obs, done, info = env.step(action)
    t = obs['sim_time']
    ego = obs['ego']
    traj.append((t, ego['x'], ego['y'], ego['v'], action['target_a']))
    for tgt in obs.get('targets', []):
        gap = math.hypot(tgt['x'] - ego['x'], tgt['y'] - ego['y'])
        min_gap = min(min_gap, gap)
        if gap < 2.5 and not collision:
            collision = True
    if done:
        break

env.close()

pet = None
if controller.t_tgt_pass is not None and controller.t_ego_pass is not None:
    pet = controller.t_ego_pass - controller.t_tgt_pass

print(f'OK: yield test done in {len(traj)} steps')
print(f'  collision={collision}, min_gap={min_gap:.2f}m')
print(f'  target pass t={controller.t_tgt_pass:.2f}s, ego pass t={controller.t_ego_pass:.2f}s')
print(f'  PET={pet:.2f}s, conflict TTC={controller.conflict_ttc:.2f}s')

## 3. 对比实验：不让行的后果 | Comparison: What Happens Without Yield

同一场景，换上 `NaiveTurnController`（直接左转，无让行逻辑）。
预期：碰撞或 PET 远小于 1.0s —— 证明让行逻辑的必要性。


In [ ]:
# Run comparison with naive controller (no yield logic)
import sys, math
sys.path.insert(0, '/home/jovyan/labs')
from common.esmini_env import EsminiEnv

naive = NaiveTurnController()
env = EsminiEnv(scenario=scenario_path, dt=0.05)
obs = env.reset(ego_x=101.5, ego_y=-30.0, ego_yaw=math.pi / 2, ego_v=9.0)
naive.reset()

traj_naive, collision_naive = [], False
min_gap_naive = float('inf')

for step in range(400):
    action = naive.step(obs)
    obs, done, info = env.step(action)
    t = obs['sim_time']
    ego = obs['ego']
    traj_naive.append((t, ego['x'], ego['y'], ego['v'], action['target_a']))
    for tgt in obs.get('targets', []):
        gap = math.hypot(tgt['x'] - ego['x'], tgt['y'] - ego['y'])
        min_gap_naive = min(min_gap_naive, gap)
        if gap < 2.5 and not collision_naive:
            collision_naive = True
    if done or collision_naive:
        break

env.close()

pet_naive = None
if naive.t_tgt_pass is not None and naive.t_ego_pass is not None:
    pet_naive = naive.t_ego_pass - naive.t_tgt_pass

print(f'OK: naive test done in {len(traj_naive)} steps')
print(f'  collision={collision_naive}, min_gap={min_gap_naive:.2f}m')
print(f'  PET={pet_naive:.2f}s, conflict TTC={naive.conflict_ttc:.2f}s')

## 4. 评测断言 | Evaluation

通过判据：让行控制器 **无碰撞** 且 **PET > 1.0s**；
对比基线应显著更差（碰撞或 PET < 1.0s）。


In [ ]:
# Pass criteria: no collision AND PET > 1.0s for the yield controller
assert not collision, 'Yield test: collision detected!'
assert pet is not None and pet > 1.0, f'Yield test: PET={pet} <= 1.0s!'
print(f'PASS: no collision, PET={pet:.2f}s > 1.0s')

# The naive baseline must be demonstrably worse (validates the experiment)
naive_unsafe = collision_naive or (pet_naive is not None and pet_naive < 1.0)
assert naive_unsafe, 'Naive baseline unexpectedly safe - check scenario setup!'
print(f'CONFIRMED: naive baseline unsafe (collision={collision_naive}, PET={pet_naive:.2f}s)')

print()
print('=== A4 LTAP-OD 结果对比 ===')
print(f'{"指标":<18}{"让行控制":<14}{"直接左转(无让行)"}')
print(f'{"是否碰撞":<18}{str(collision):<14}{collision_naive}')
print(f'{"PET (s)":<18}{pet:<14.2f}{pet_naive:.2f}')
print(f'{"冲突点TTC (s)":<18}{controller.conflict_ttc:<14.2f}{naive.conflict_ttc:.2f}')
print(f'{"最小车间距 (m)":<18}{min_gap:<14.2f}{min_gap_naive:.2f}')

## 六、真实仿真轨迹可视化 | Visualization

俯视图：Ego 左转轨迹 vs Target 直行轨迹，标注冲突点与停车线。


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- left: top-down trajectories ---
ax = axes[0]
ex = [p[1] for p in traj]; ey = [p[2] for p in traj]
nx = [p[1] for p in traj_naive]; ny = [p[2] for p in traj_naive]
ax.plot(ex, ey, 'b-', lw=2, label='Ego (yield)')
ax.plot(nx, ny, 'r--', lw=1.5, label='Ego (naive)')
# target lane (straight southbound at x=98.5)
ax.plot([98.5, 98.5], [-35, 60], 'g-', lw=2, label='Target lane')
ax.plot([CX], [CY], 'kx', ms=12, mew=2, label='Conflict point')
ax.plot([95, 108], [STOP_Y, STOP_Y], 'm--', lw=1.5, label='Stop line')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Top-down trajectories (real esmini)')
ax.legend(fontsize=8); ax.axis('equal'); ax.grid(True, alpha=0.3)

# --- right: ego speed profiles ---
ax = axes[1]
ax.plot([p[0] for p in traj], [p[3] for p in traj], 'b-', label='Ego speed (yield)')
ax.plot([p[0] for p in traj_naive], [p[3] for p in traj_naive], 'r--', label='Ego speed (naive)')
ax.axvline(controller.t_tgt_pass or 0, color='g', ls=':', label='Target passes conflict')
ax.set_xlabel('time (s)'); ax.set_ylabel('speed (m/s)')
ax.set_title('Ego speed: yield vs naive')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 七、🏆 挑战任务 | Challenges

1. **激进驾驶员**：把 `margin` 从 1.5s 改为 0.5s，重新运行。PET 降到多少？是否仍通过？
2. **高速来车**：修改 xosc 中 Target 速度为 20 m/s（同步更新 `TGT_V` 假设），让行逻辑需要调整哪个参数？
3. **行人变体**：在冲突点附近增加一个低速横穿的行人目标（esmini 行人实体），扩展为三参与者让行。
4. **真实数据链**：用 `common.vehicle_library.save_result('A4_intersection_yield', {...})` 保存本次的 PET/冲突点TTC，供 E3（MPC 控制器）作为基线对比。


---

## 下一个实验 | Next →

**[A5_merge_kinematic_demo](./A5_merge_kinematic_demo.ipynb)**

> **所需内核**：`python3` | **Required kernel**: `python3`
